# Upload the datasets for train (Save from the feature engineering code)

In [1]:
import pandas as pd

# 📌 Catboost training without gridsearch and feature selection
- Second feature engineering method used for the following two trains

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from math import sqrt

ID_COL = "student_id"
RANDOM_STATE = 42
TARGET = "career_success_score"

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# CatBoost parametreleri
cat_params = {
    "loss_function": "RMSE",
    "eval_metric": "RMSE",
    "iterations": 1500,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 5,
    "random_seed": RANDOM_STATE,
    "od_type": "Iter",
    "od_wait": 100,
    "verbose": 200,
    "allow_writing_files": False
}

# KFold CV
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), start=1):
    print(f"========== Fold {fold} ==========")

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostRegressor(**cat_params)
    model.fit(
        X_train, y_train,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    valid_pred = model.predict(X_valid)
    oof_preds[valid_idx] = valid_pred

    fold_rmse = mean_squared_error(y_valid, valid_pred)
    fold_rmse = sqrt(fold_rmse)
    fold_scores.append(fold_rmse)
    print(f"Fold {fold} RMSE: {fold_rmse:.5f}")

    test_preds += model.predict(X_test) / kf.n_splits

# OOF RMSE
oof_rmse = mean_squared_error(y, oof_preds)
oof_rmse = sqrt(oof_rmse)
print("========== CV RESULT ==========")
print(f"Fold RMSE values: {[round(s, 5) for s in fold_scores]}")
print(f"Mean RMSE: {np.mean(fold_scores):.5f}")
print(f"Std RMSE : {np.std(fold_scores):.5f}")
print(f"OOF RMSE : {oof_rmse:.5f}")

# Tahminleri 0-100 aralığına sıkıştır
test_preds_clipped = np.clip(test_preds, 0, 100)

submission = pd.DataFrame({
    ID_COL: test_df[ID_COL],
    TARGET: test_preds_clipped
})

submission.to_csv("catboost_submission.csv", index=False)
print("Saved: catboost_submission.csv")

# 📌 GridSearch ile CatBoost Parametre Optimizasyonu

In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, mean_squared_error
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd
from math import sqrt

ID_COL = "student_id"
RANDOM_STATE = 42
TARGET = "career_success_score"

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(X_test)

# CatBoost model (base)
cat_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    verbose=0,
    allow_writing_files=False
)

# GridSearch parametreleri
param_grid = {
    "depth": [4, 6, 8],
    "learning_rate": [0.01, 0.03, 0.05],
    "l2_leaf_reg": [3, 5, 7],
    "iterations": [500, 1000, 1500]
}

# RMSE scorer
rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False)

# KFold CV
kf = KFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# GridSearch
grid_search = GridSearchCV(
    estimator=cat_model,
    param_grid=param_grid,
    scoring=rmse_scorer,
    cv=kf,
    n_jobs=-1
)

grid_search.fit(X_scaled, y)

print("Best Parameters:", grid_search.best_params_)
print("Best RMSE (MSE value, take sqrt):", sqrt(-grid_search.best_score_))
